# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [46]:
from pathlib import Path
import os

import faiss
import numpy as np
import pandas as pd
from google import genai
from google.genai import types

api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise EnvironmentError('Set the GEMINI_API_KEY environment variable before running this notebook.')
client = genai.Client(api_key=api_key)

csv_name = 'ecommerce_faq_knowledge_base.csv'
csv_candidates = [
    Path.cwd() / 'data' / csv_name,
    Path.cwd().parent / 'data' / csv_name,
    Path.cwd() / csv_name,
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f'Could not find {csv_name}. Expected it in the project data/ folder.')

df = pd.read_csv(csv_path)
print('Dataset:', csv_path)
print('Total entri knowledge base:', len(df))
print('Kolom:', ', '.join(df.columns))
df.head()

Dataset: c:\Users\acer\Documents\semantic-search-assignment\data\ecommerce_faq_knowledge_base.csv
Total entri knowledge base: 59
Kolom: faq_id, category, question, answer


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [48]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Setiap FAQ diubah menjadi vektor menggunakan Gemini Embeddings API. Data FAQ memakai task type `RETRIEVAL_DOCUMENT`, sedangkan pertanyaan pengguna nantinya memakai `RETRIEVAL_QUERY`. Dimensi 768 dipilih agar ukuran vektor lebih ringan; yang penting, dimensi untuk dokumen dan query harus sama.

Karena dataset ini kecil, embedding seluruh FAQ dibuat sekaligus. Periksa bentuk matriksnya sebelum melanjutkan: jumlah baris harus sama dengan jumlah FAQ, dan jumlah kolom menunjukkan dimensi vektor.

In [50]:
EMBEDDING_MODEL = 'gemini-embedding-001'
EMBEDDING_DIMENSION = 768

def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBEDDING_DIMENSION,
        ),
    )
    return result.embeddings[0].values

embedding_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=documents,
    config=types.EmbedContentConfig(
        task_type='RETRIEVAL_DOCUMENT',
        output_dimensionality=EMBEDDING_DIMENSION,
    ),
)
embeddings = np.asarray(
    [item.values for item in embedding_response.embeddings], dtype='float32'
)

if embeddings.shape != (len(documents), EMBEDDING_DIMENSION):
    raise ValueError(f'Unexpected embedding shape: {embeddings.shape}')
if not np.isfinite(embeddings).all():
    raise ValueError('Embeddings contain non-finite values.')

print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 768)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

Vektor dokumen dinormalisasi terlebih dahulu. Setelah normalisasi, inner product dua vektor setara dengan cosine similarity, sehingga `IndexFlatIP` dapat dipakai untuk mengurutkan FAQ yang paling dekat dengan query. Pastikan `index.ntotal` sama dengan jumlah entri knowledge base.

In [51]:
dimension = embeddings.shape[1]
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
assert index.ntotal == len(df), 'Jumlah vektor harus sama dengan jumlah FAQ.'

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

FAISS mengembalikan skor dan nomor baris, bukan isi FAQ. Fungsi di bawah menghubungkan nomor baris itu kembali ke data asli, lalu menampilkan pertanyaan, jawaban, kategori, dan skor similarity. Karena index memakai inner product pada vektor yang sudah dinormalisasi, skor yang lebih tinggi berarti hasilnya lebih mirip.

In [52]:
def semantic_search(query, top_k=3):
    if not query or not query.strip():
        raise ValueError('Query must not be empty.')
    if top_k < 1:
        raise ValueError('top_k must be at least 1.')

    query_embedding = np.asarray([get_embedding(query)], dtype='float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, min(top_k, len(df)))

    results = []
    for rank, (similarity, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        if idx < 0:
            continue
        results.append({
            'rank': rank,
            'faq_id': int(df.iloc[idx]['faq_id']),
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'similarity': float(similarity),
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for result in test_results:
    print(f"{result['similarity']:.3f} | {result['question']}")

0.800 | Bagaimana jika paket saya belum sampai melebihi estimasi?
0.781 | Kenapa status pengiriman saya tidak berubah selama beberapa hari?
0.759 | Kenapa nomor resi saya belum bisa dilacak di website kurir?


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [58]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {result['question']}\nJawaban: {result['answer']}" for result in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [67]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h4m34.478951818s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flas

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (parafrase/sinonim). Untuk query yang jawabannya ada, bandingkan FAQ teratas dengan jawaban yang seharusnya ditemukan. Untuk query di luar knowledge base, periksa apakah jawabannya mengakui informasi belum tersedia.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di knowledge base_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [63]:
test_cases = [
    {
        'query': 'Berapa lama batas waktu pembayaran sebelum pesanan dibatalkan?',
        'expected_faq_id': 19,
        'type': 'langsung',
    },
    {
        'query': 'Saya sudah checkout, apakah alamat tujuan masih bisa diganti?',
        'expected_faq_id': 5,
        'type': 'parafrasa',
    },
    {
        'query': 'Paket saya belum datang padahal estimasi sudah lewat, apa yang harus saya lakukan?',
        'expected_faq_id': 12,
        'type': 'parafrasa',
    },
    {
        'query': 'Bisa bayar pakai cicilan kartu kredit?',
        'expected_faq_id': 20,
        'type': 'langsung',
    },
    {
        'query': 'Apakah tersedia jasa pemasangan perabot di rumah?',
        'expected_faq_id': None,
        'type': 'di luar knowledge base',
    },
]

evaluation_results = []
for case in test_cases:
    sources = semantic_search(case['query'])
    top_source = sources[0] if sources else None
    result = {
        **case,
        'top_faq_id': top_source['faq_id'] if top_source else None,
        'top_similarity': top_source['similarity'] if top_source else None,
        'retrieval_hit': (
            top_source is not None
            and case['expected_faq_id'] is not None
            and top_source['faq_id'] == case['expected_faq_id']
        ),
        'answer': None,
        'llm_error': None,
    }

    try:
        result['answer'], _ = generate_answer(case['query'])
    except Exception as error:
        result['llm_error'] = f'{type(error).__name__}: {error}'

    evaluation_results.append(result)
    print(f"[{case['type']}] {case['query']}")
    print(f"  Retrieval: FAQ {result['top_faq_id']} | similarity={result['top_similarity']:.3f}" if top_source else '  Retrieval: tidak ada hasil')
    print(f"  Hit FAQ yang diharapkan: {result['retrieval_hit']}")
    print(f"  Jawaban: {result['answer'] or result['llm_error']}")
    print()


[langsung] Berapa lama batas waktu pembayaran sebelum pesanan dibatalkan?
  Retrieval: FAQ 19 | similarity=0.866
  Hit FAQ yang diharapkan: True
  Jawaban: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h9m26.043794236s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_cont

### Catatan Analisis Relevansi

Dari lima pertanyaan yang diuji, empat pertanyaan yang jawabannya ada di FAQ berhasil menemukan entri yang sesuai di urutan pertama. Pertanyaan tentang batas waktu pembayaran menemukan FAQ 19 dengan similarity `0.866`; pertanyaan tentang perubahan alamat setelah checkout menemukan FAQ 5 (`0.793`); pertanyaan tentang paket yang terlambat menemukan FAQ 12 (`0.831`); dan pertanyaan tentang cicilan kartu kredit menemukan FAQ 20 (`0.823`). Pada contoh ini, parafrasa tetap membawa pencarian ke FAQ yang benar.

Pertanyaan tentang jasa pemasangan perabot tidak ada di knowledge base. FAISS tetap mengembalikan FAQ terdekat, yaitu FAQ 15 dengan similarity `0.663`. Kode saat ini tidak memakai cutoff, sehingga FAQ tersebut tetap diberikan sebagai konteks ke Gemini. Prompt meminta model menjawab hanya berdasarkan FAQ dan mengakui jika informasi tidak tersedia, tetapi respons untuk kasus ini belum berhasil diverifikasi.

Pada pengujian terakhir, kelima permintaan Gemini gagal dengan `429 RESOURCE_EXHAUSTED` karena kuota gratis harian `gemini-3.8-flash` telah tercapai. Jadi, hasil retrieval dapat dianalisis, tetapi kualitas jawaban LLM belum dapat disimpulkan dari pengujian terakhir.